In [1]:
"""
DDIM on CIFAR-10 (PyTorch)
-------------------------------------------------
- Trains a standard DDPM-style epsilon-prediction U-Net on CIFAR-10
- Adds a DDIM sampler (deterministic when eta=0)

Usage (GPU recommended):
  python ddim_cifar10.py --epochs 200 --batch-size 128 --save-dir runs/ddim

This is a single-file, minimal-yet-solid reference. Uses a light U-Net suitable for 32x32.
"""
import math
import os
import argparse
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils

# ----------------------------
# Helpers
# ----------------------------

def exists(x):
    return x is not None


def default(val, d):
    return val if exists(val) else d


# Sinusoidal timestep embedding (like Transformer positional embedding)
def timestep_embedding(timesteps: torch.Tensor, dim: int, max_period: int = 10000):
    """Create sinusoidal timestep embeddings.
    timesteps: (B,) int or float tensor in [0, T-1]
    returns: (B, dim)
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half
    )  # (half,)
    args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)  # (B, half)
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2 == 1:  # pad if needed
        emb = F.pad(emb, (0, 1))
    return emb


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch)
        )

        self.norm2 = nn.GroupNorm(32, out_ch)
        self.act2 = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act1(self.norm1(x)))
        # FiLM-style time conditioning: add projected time embedding
        h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(self.dropout(self.act2(self.norm2(h))))
        return h + self.skip(x)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, 4, 2, 1)

    def forward(self, x):
        return self.op(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.ConvTranspose2d(ch, ch, 4, 2, 1)

    def forward(self, x):
        return self.op(x)


class SelfAttention(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(32, ch)
        self.num_heads = num_heads
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        hdim = c // self.num_heads
        x_norm = self.norm(x)
        q, k, v = self.qkv(x_norm).chunk(3, dim=1)
        # reshape to heads
        q = q.reshape(b, self.num_heads, hdim, h * w)
        k = k.reshape(b, self.num_heads, hdim, h * w)
        v = v.reshape(b, self.num_heads, hdim, h * w)
        attn = torch.softmax((q.transpose(-2, -1) @ k) / math.sqrt(hdim), dim=-1)  # (b, heads, hw, hw)
        out = (attn @ v.transpose(-2, -1)).transpose(-2, -1)  # (b, heads, hdim, hw)
        out = out.reshape(b, c, h, w)
        return x + self.proj(out)


class UNet32(nn.Module):
    """A light U-Net backbone for 32x32 images that predicts epsilon.
    Conditional only on time t.
    """
    def __init__(self, in_ch=3, base_ch=128, time_dim=256, ch_mults=(1, 2, 2, 2), dropout=0.0):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim)
        )

        chs = [base_ch * m for m in ch_mults]

        # Enc
        self.in_conv = nn.Conv2d(in_ch, chs[0], 3, padding=1)
        self.down1 = ResidualBlock(chs[0], chs[0], time_dim, dropout)
        self.down1_attn = SelfAttention(chs[0])
        self.down1_ds = Downsample(chs[0])  # 32->16

        self.down2 = ResidualBlock(chs[0], chs[1], time_dim, dropout)
        self.down2_attn = SelfAttention(chs[1])
        self.down2_ds = Downsample(chs[1])  # 16->8

        self.down3 = ResidualBlock(chs[1], chs[2], time_dim, dropout)
        self.down3_attn = SelfAttention(chs[2])
        self.down3_ds = Downsample(chs[2])  # 8->4

        # Bottleneck
        self.mid1 = ResidualBlock(chs[2], chs[3], time_dim, dropout)
        self.mid_attn = SelfAttention(chs[3])
        self.mid2 = ResidualBlock(chs[3], chs[3], time_dim, dropout)

        # Dec
        self.up3_us = Upsample(chs[3])      # 4->8
        self.up3 = ResidualBlock(chs[3] + chs[2], chs[2], time_dim, dropout)
        self.up3_attn = SelfAttention(chs[2])

        self.up2_us = Upsample(chs[2])      # 8->16
        self.up2 = ResidualBlock(chs[2] + chs[1], chs[1], time_dim, dropout)
        self.up2_attn = SelfAttention(chs[1])

        self.up1_us = Upsample(chs[1])      # 16->32
        self.up1 = ResidualBlock(chs[1] + chs[0], chs[0], time_dim, dropout)
        self.up1_attn = SelfAttention(chs[0])

        self.out_norm = nn.GroupNorm(32, chs[0])
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(chs[0], in_ch, 3, padding=1)

    def forward(self, x, t):
        # t: (B,) or (B,1)
        if t.dim() == 0:
            t = t[None]
        if t.dim() == 1:
            t = t[:, None]
        t_emb = timestep_embedding(t.squeeze(-1), self.time_dim)
        t_emb = self.time_mlp(t_emb)

        # Encoder
        x0 = self.in_conv(x)
        d1 = self.down1(x0, t_emb); d1 = self.down1_attn(d1); d1_ds = self.down1_ds(d1)
        d2 = self.down2(d1_ds, t_emb); d2 = self.down2_attn(d2); d2_ds = self.down2_ds(d2)
        d3 = self.down3(d2_ds, t_emb); d3 = self.down3_attn(d3); d3_ds = self.down3_ds(d3)

        # Mid
        m = self.mid1(d3_ds, t_emb); m = self.mid_attn(m); m = self.mid2(m, t_emb)

        # Decoder
        u3 = self.up3_us(m)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.up3(u3, t_emb); u3 = self.up3_attn(u3)

        u2 = self.up2_us(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.up2(u2, t_emb); u2 = self.up2_attn(u2)

        u1 = self.up1_us(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.up1(u1, t_emb); u1 = self.up1_attn(u1)

        out = self.out_conv(self.out_act(self.out_norm(u1)))
        return out  # predicts epsilon with shape like x


# ----------------------------
# Noise schedule utilities
# ----------------------------

def make_beta_schedule(T: int, schedule: str = "linear", beta_start=1e-4, beta_end=2e-2):
    if schedule == "linear":
        return torch.linspace(beta_start, beta_end, T)
    elif schedule == "cosine":
        # Nichol & Dhariwal cosine schedule for alpha_bar
        s = 0.008
        steps = torch.arange(T + 1, dtype=torch.float32)
        alphas_cumprod = torch.cos(((steps / T) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clamp(betas, 1e-5, 0.999)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")


@dataclass
class DiffusionSchedule:
    T: int
    betas: torch.Tensor
    alphas: torch.Tensor
    alphas_cumprod: torch.Tensor
    sqrt_alphas_cumprod: torch.Tensor
    sqrt_one_minus_alphas_cumprod: torch.Tensor


def build_schedule(T: int, schedule: str = "linear", device: torch.device = torch.device("cpu")):
    betas = make_beta_schedule(T, schedule).to(device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return DiffusionSchedule(
        T=T,
        betas=betas,
        alphas=alphas,
        alphas_cumprod=alphas_cumprod,
        sqrt_alphas_cumprod=torch.sqrt(alphas_cumprod),
        sqrt_one_minus_alphas_cumprod=torch.sqrt(1.0 - alphas_cumprod),
    )


# ----------------------------
# DDIM Sampler
# ----------------------------

def make_ddim_timesteps(T: int, steps: int):
    """Return a tensor of length `steps` with indices in [0, T-1], descending.
    (E.g., T=1000, steps=50 -> 999, 979, ..., 19, 0)
    """
    return torch.linspace(T - 1, 0, steps, dtype=torch.long)


def ddim_step(model: nn.Module, x_t: torch.Tensor, t: int, t_prev: int,
              alphas_cumprod: torch.Tensor, eta: float = 0.0):
    """One DDIM update from x_t at time t -> x_{t_prev}.
    - model predicts epsilon
    - when eta=0, this is deterministic (probability flow ODE)
    """
    device = x_t.device
    a_bar_t = alphas_cumprod[t]
    a_bar_prev = alphas_cumprod[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)

    # predict epsilon
    t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
    eps_hat = model(x_t, t_batch)

    # predict x0
    x0_hat = (x_t - torch.sqrt(1.0 - a_bar_t) * eps_hat) / torch.sqrt(a_bar_t)

    # DDIM variance
    if t_prev >= 0:
        sigma_t = (
            eta
            * torch.sqrt((1 - a_bar_prev) / (1 - a_bar_t))
            * torch.sqrt(1 - a_bar_t / a_bar_prev)
        )
    else:
        sigma_t = torch.tensor(0.0, device=device)

    # direction pointing to x_{t-1}
    dir_xt = torch.sqrt(1.0 - a_bar_prev - sigma_t**2) * eps_hat if t_prev >= 0 else 0.0
    noise = sigma_t * torch.randn_like(x_t) if t_prev >= 0 else 0.0

    x_prev = torch.sqrt(a_bar_prev) * x0_hat + dir_xt + noise
    return x_prev.clamp(-1, 1)


def ddim_sample(model: nn.Module, schedule: DiffusionSchedule, shape, steps: int = 50, eta: float = 0.0):
    model.eval()
    device = next(model.parameters()).device
    x = torch.randn(shape, device=device)

    ts = make_ddim_timesteps(schedule.T, steps).to(device)
    for i, t in enumerate(ts):
        t = t.item()
        t_prev = ts[i + 1].item() if i + 1 < len(ts) else -1
        x = ddim_step(model, x, t, t_prev, schedule.alphas_cumprod, eta)
    return x


# ----------------------------
# EMA (Exponential Moving Average) for weights
# ----------------------------
class EMA:
    def __init__(self, model, beta=0.9999):
        self.beta = beta
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.detach().clone()

    @torch.no_grad()
    def update(self, model):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            assert name in self.shadow
            self.shadow[name].mul_(self.beta).add_(param.detach(), alpha=1 - self.beta)

    @torch.no_grad()
    def copy_to(self, model):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            param.data.copy_(self.shadow[name].data)


# ----------------------------
# Data utils
# ----------------------------

def get_cifar10_loaders(batch_size=128, num_workers=4, data_dir="./data"):
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2.0 - 1.0),  # [-1,1]
    ])
    train_ds = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=tfm)
    test_ds = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=tfm)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader


# ----------------------------
# Training
# ----------------------------

def loss_ddpm_epsilon(model: nn.Module, x0: torch.Tensor, schedule: DiffusionSchedule):
    B = x0.size(0)
    device = x0.device
    t = torch.randint(0, schedule.T, (B,), device=device, dtype=torch.long)

    a_bar_t = schedule.alphas_cumprod[t].view(B, 1, 1, 1)
    one_minus = schedule.sqrt_one_minus_alphas_cumprod[t].view(B, 1, 1, 1)

    eps = torch.randn_like(x0)
    x_t = schedule.sqrt_alphas_cumprod[t].view(B,1,1,1) * x0 + one_minus * eps

    eps_pred = model(x_t, t)
    return F.mse_loss(eps_pred, eps)


def save_image_grid(x, path, nrow=8):
    x = (x.clamp(-1, 1) + 1) * 0.5  # to [0,1]
    vutils.save_image(x, path, nrow=nrow)

In [2]:
# parser.add_argument("--epochs", type=int, default=100)
# parser.add_argument("--batch-size", type=int, default=128)
# parser.add_argument("--lr", type=float, default=2e-4)
# parser.add_argument("--beta1", type=float, default=0.9)
# parser.add_argument("--beta2", type=float, default=0.999)
# parser.add_argument("--num-workers", type=int, default=4)
# parser.add_argument("--T", type=int, default=1000)
# parser.add_argument("--schedule", type=str, default="linear", choices=["linear", "cosine"])
# parser.add_argument("--save-dir", type=str, default="runs/ddim")
# parser.add_argument("--sample-every", type=int, default=1000, help="iters between samples")
# parser.add_argument("--ddim-steps", type=int, default=50)
# parser.add_argument("--ddim-eta", type=float, default=0.0)
# parser.add_argument("--no-ema", action="store_true")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("./data", exist_ok=True)

# Data
train_loader, _ = get_cifar10_loaders(128, 0)

# Model & schedule
model = UNet32(in_ch=3, base_ch=128, time_dim=256, ch_mults=(1, 2, 2, 2)).to(device)
schedule = build_schedule(1000, "linear", device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, betas=(0.9, 0.999))
scaler = torch.amp.GradScaler('cuda',enabled=False)#, enabled=torch.cuda.is_available())
ema = EMA(model)
# ema = None if args.no-ema else EMA(model)
global_step = 0
for epoch in range(10):
    for x, _ in train_loader:
        model.train()
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=False):#, enabled=torch.cuda.is_available()):
            loss = loss_ddpm_epsilon(model, x, schedule)
        # scaler.scale(loss).backward()
        # scaler.step(optimizer)
        # scaler.update()
        
        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)

        if global_step % 10 == 0:
            print(f"epoch {epoch} step {global_step} loss {loss.item():.4f}")

        # Sampling preview
        if global_step % 1000 == 0:
            model.eval()
            with torch.no_grad():
                model_for_sample = model
                if ema is not None:
                    # copy EMA weights into a temp model
                    shadow_model = UNet32(in_ch=3, base_ch=128, time_dim=256, ch_mults=(1,2,2,2)).to(device)
                    shadow_model.load_state_dict(model.state_dict(), strict=True)
                    ema.copy_to(shadow_model)
                    model_for_sample = shadow_model

                imgs = ddim_sample(
                    model_for_sample,
                    schedule,
                    shape=(64, 3, 32, 32),
                    steps=50,
                    eta=0,
                )
                save_image_grid(imgs, os.path.join("./data", f"samples_{global_step:07d}.png"), nrow=8)
                # also save a small set of noised inputs -> optional

        global_step += 1

    # end epoch: checkpoint
    ckpt = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
        'global_step': global_step
    }
    if ema is not None:
        ckpt['ema'] = {k: v.cpu() for k, v in ema.shadow.items()}
    torch.save(ckpt, os.path.join("./data", f"ckpt_epoch_{epoch:03d}.pt"))

epoch 0 step 0 loss 1.1099
epoch 0 step 10 loss 4.3497


KeyboardInterrupt: 